# SMS Shield: Exploratory Data Analysis & Drift Baseline Study

This notebook performs full Exploratory Data Analysis (EDA) on the SMS Spam Collection dataset to establish baseline feature distributions for our unsupervised SMS drift monitoring system.

## Key Goals:
1. Understand dataset composition, missing values, and class imbalance (Ham vs Spam).
2. Analyze domain-specific text properties (message length, digit count, uppercase ratio, special character ratio).
3. Evaluate Character N-Gram TF-IDF (range 3-5) for capturing obfuscations (e.g. `fr33`, `cl1ck`, `33397$`).
4. Compare TruncatedSVD vs PCA for sparse matrix reduction.


In [ ]:
import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD

# Set aesthetics
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('muted')


## 1. Load Dataset & Schema Inspection

In [ ]:
df = pd.read_csv('../data/raw/sms_dataset.csv')
print('Dataset Shape:', df.shape)
print('Missing values:
', df.isnull().sum())
df.head()


## 2. Class Imbalance Analysis

In [ ]:
counts = df['label'].value_counts()
print('Class counts:
', counts)
print('Ham percentage: {:.2f}%'.format((counts['ham'] / len(df)) * 100))
print('Spam percentage: {:.2f}%'.format((counts['spam'] / len(df)) * 100))


## 3. Feature Extraction (Message Length, Digits, Upper & Special Char Ratios)

In [ ]:
df['char_length'] = df['message'].apply(len)
df['digit_count'] = df['message'].apply(lambda x: sum(c.isdigit() for c in x))
df['upper_ratio'] = df['message'].apply(lambda x: sum(c.isupper() for c in x)) / (df['char_length'] + 1e-5)
df['special_char_ratio'] = df['message'].apply(lambda x: len(re.findall(r'[^a-zA-Z0-9\s]', x))) / (df['char_length'] + 1e-5)

df.groupby('label')[['char_length', 'digit_count', 'upper_ratio', 'special_char_ratio']].mean()


## 4. Character N-Gram TF-IDF Vectorization (3,5)

In [ ]:
vec = TfidfVectorizer(analyzer='char', ngram_range=(3, 5), max_features=5000)
tfidf = vec.fit_transform(df['message'].str.lower())
print('TF-IDF Sparse Matrix Shape:', tfidf.shape)


## 5. Justification: TruncatedSVD vs PCA for Sparse Matrices


### Why TruncatedSVD is Preferred over PCA:
- **Preserves Sparsity**: Standard PCA centers data by subtracting the mean, which converts a sparse matrix (mostly zeros) into a dense matrix requiring immense memory ((N \cdot D)$).
- **Efficiency**: TruncatedSVD works directly on sparse SciPy matrices using fast randomized SVD solvers without dense conversion.


In [ ]:
svd = TruncatedSVD(n_components=100, random_state=42)
X_svd = svd.fit_transform(tfidf)
print('TruncatedSVD Explained Variance Ratio Sum (100 components):', np.sum(svd.explained_variance_ratio_))
